### Data Ingestion


In [14]:
###Document Structure
from langchain_community.document_loaders import DirectoryLoader, TextLoader

loader = DirectoryLoader(
    "../knowledge",
    glob="**/*.md",
    loader_cls=TextLoader,
    show_progress=True
    )
    
documents = loader.load()
texts = [doc.page_content for doc in documents]

100%|██████████| 13/13 [00:00<00:00, 4360.33it/s]


In [15]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150
)

chunks = text_splitter.split_documents(documents)

print(f"Original documents: {len(documents)}")
print(f"Total chunks: {len(chunks)}")

texts = [chunk.page_content for chunk in chunks]

Original documents: 13
Total chunks: 82


### Embedding and VectorStoreDB

In [16]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [17]:
class EmbeddingManager:
    """Handles document embedding generation using Sentence Transformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager.

        Args:
            model_name: HuggingFace model name for sentence transformer.
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the Sentence Transformer."""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Embedding model loaded successfully. Dimendions : {self.model.get_embedding_dimension()}")
        except Exception as error:
            raise RuntimeError(
                f"Failed to load embedding model '{self.model_name}'."
            ) from error

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for list of texts

        Args:
            texts : List of text strings to embed

        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")

        print(f"Generating embeddings for {len(texts)} texts..")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape : {embeddings.shape}")
        return embeddings

# initialize the embedding manager
embedding_manager = EmbeddingManager()
embeddings = embedding_manager.generate_embeddings(texts)

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11130.63it/s]


Embedding model loaded successfully. Dimendions : 384
Generating embeddings for 82 texts..


Batches: 100%|██████████| 3/3 [00:01<00:00,  2.20it/s]

Generated embeddings with shape : (82, 384)


### VectorStore

In [20]:
import os
class VectorStore:
    """
    Manages document embeddings in a ChromaDB vector store 
    """
    def __init__(
            self,
            collection_name : str="portfolio_knowledge",
            persist_directory: str="../knowledge/vector_store"
    ):
        """ 
        Initialize the Vector Store

        Args:
            collection_name : Name of the chromaDB collection
            persist_directory : Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection."""
        try:
            # Create persistent ChromaDB directory
            os.makedirs(
                self.persist_directory,
                exist_ok=True
            )
            # Create persistent ChromaDB client
            self.client = chromadb.PersistentClient(
                path=self.persist_directory
            )
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={
                    "description": "Portfolio Markdown knowledge embeddings for RAG"
                }
            )
            print(
                f"Vector store initialized. "
                f"Collection: {self.collection_name}"
            )

            print(
                f"Existing documents in collection: "
                f"{self.collection.count()}"
            )
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(
        self,
        documents: List[Any],
        embeddings: np.ndarray
    ):
        """
        Add documents and their embeddings to the vector store.

        Args:
            documents: List of LangChain documents.
            embeddings: Corresponding embeddings for the documents.
        """
        # Make sure the number of documents matches
        # the number of embeddings
        if len(documents) != len(embeddings):
            raise ValueError(
                "Number of documents must match number of embeddings"
            )
        print(
            f"Adding {len(documents)} documents to vector store..."
        )
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        # Process each document and its corresponding embedding
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            source = doc.metadata.get("source", "unknown")
            doc_id = f"{source}_{i}".replace("\\", "_").replace("/", "_")
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)

            metadatas.append(metadata)

            # Document content
            documents_text.append(doc.page_content)

            # Embedding
            embeddings_list.append(embedding.tolist())

        # Add documents to ChromaDB collection
        try:
            self.collection.upsert(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(
                f"Successfully added {len(documents)} "
                f"documents to vector store"
            )
            print(
                f"Total documents in collection: "
                f"{self.collection.count()}"
            )

        except Exception as e:
            print(
                f"Error adding documents to vector store: {e}"
            )
            raise
vectorstore = VectorStore()

Vector store initialized. Collection: portfolio_knowledge
Existing documents in collection: 82


In [19]:
# convert the chunks to embeddings
texts = [chunk.page_content for chunk in chunks]

# generate the embeddings
embeddings = embedding_manager.generate_embeddings(texts)

# store in the vector database
vectorstore.add_documents(chunks, embeddings)

Generating embeddings for 82 texts..


Batches: 100%|██████████| 3/3 [00:01<00:00,  2.06it/s]

Generated embeddings with shape : (82, 384)
Adding 82 documents to vector store...
Successfully added 82 documents to vector store
Total documents in collection: 82


### Retriever Pipeline from VectorStore

In [27]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever.

        Args:
            vector_store: Vector store containing document embeddings.
            embedding_manager: Manager for generating query embeddings.
        """

        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query.

        Args:
            query: The search query.
            top_k: Number of top results to return.
            score_threshold: Minimum similarity score threshold.

        Returns:
            List of dictionaries containing retrieved documents and metadata.
        """

        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, "f"Score threshold: {score_threshold}")

        try:
            # Generate query embedding
            query_embedding = (self.embedding_manager.generate_embeddings([query])[0])

            # Search in vector store
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            # Process results
            retrieved_docs = []
            if results['documents'] and results['documents'][0]:

                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(
                    zip(ids, documents, metadatas, distances)
                ):
                    # Convert distance to similarity score
                    # ChromaDB uses cosine distance
                    similarity_score = 1 - distance

                    # Apply similarity threshold
                    if similarity_score >= score_threshold:

                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })

                print( f"Retrieved {len(retrieved_docs)} " f"documents (after filtering)")
            else:
                print("No documents found")

            return retrieved_docs

        except Exception as e:
            print(
                f"Error during retrieval: {e}"
            )
            return []
# instantiate using the existing notebook variables
rag_retriever = RAGRetriever(vectorstore, embedding_manager)


In [35]:
rag_retriever.retrieve("what are the skills abhishek have")

Retrieving documents for query: 'what are the skills abhishek have'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts..


Batches: 100%|██████████| 1/1 [00:00<00:00, 60.85it/s]

Generated embeddings with shape : (1, 384)
Retrieved 3 documents (after filtering)


[{'id': '.._knowledge_resume.md_52',
  'content': "---\n\n## Certifications\n- **GeeksforGeeks Onsite Workshop** â€“ Advanced DSA and competitive programming strategies  \n- **HCL GUVI AI Impact Summit** â€“ Real-world AI/ML applications and responsible AI deployment  \n- **Azisly AI Live Project** â€“ Consulting-driven AI business simulation: prompt engineering, AI-based research, corporate reporting  \n\n---\n\n## Target RAG Retrieval Queries\nThis document provides comprehensive answers to queries such as:\n- Can you provide Abhishek Yadav's complete resume?\n- Give me a full summary of Abhishek Yadav's credentials, education, projects, and skills.\n- What are all the sections in Abhishek Yadav's resume?",
  'metadata': {'content_length': 652,
   'doc_index': 52,
   'source': '..\\knowledge\\resume.md'},
  'similarity_score': 0.05407071113586426,
  'distance': 0.9459292888641357,
  'rank': 1},
 {'id': '.._knowledge_projects.md_38',
  'content': "---\n\n## Target RAG Retrieval Querie